# Colab GPU training — deep models (TCN; later Bi-LSTM / uni-LSTM)

Runs the **same `src/` code** as the local repo on a free T4 GPU. Nothing here is model logic — every cell
calls a script from the repo. Full explanation: `docs/COLAB.md`.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.
Outputs go to Google Drive (`MyDrive/pm-outputs/`), so a disconnect loses nothing: re-run the notebook and the
Optuna study **resumes** where it stopped.

## 1 · GPU check — must print a T4

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 · Google Drive (persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
OUT = pathlib.Path('/content/drive/MyDrive/pm-outputs'); OUT.mkdir(parents=True, exist_ok=True)
print(OUT)

## 3 · Code: clone the public repo (fresh copy each session)

In [ ]:
%cd /content
!rm -rf predictive-maintenance
!git clone -q https://github.com/syed-m07/predictive-maintenance.git
%cd /content/predictive-maintenance
!git log --oneline -1

## 4 · Dependencies **without torch**
Colab already has a CUDA build of PyTorch. Our `requirements.txt` pins the CPU build, which would replace it and
silently disable the GPU — so torch lines are filtered out. If pip reports conflicts, *Runtime → Restart session*
once and re-run from step 3.

In [ ]:
!grep -v -E "^(torch|--extra-index-url)" requirements.txt > /tmp/req-colab.txt
!pip install -q -r /tmp/req-colab.txt
import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU not available - check the runtime type'

## 5 · Data (same Kaggle source and checksums as local)

In [ ]:
import kagglehub, shutil, glob
src = kagglehub.dataset_download('behrad3d/nasa-cmaps') + '/CMaps'
for f in glob.glob(src + '/*_FD00?.txt'):
    shutil.copy(f, 'data/raw/')
!cd data/raw && sha256sum -c ../sha256sums.txt --ignore-missing

## 6 · Point MLflow and Optuna at Drive
So runs and the study survive the session.

In [ ]:
os.environ['MLFLOW_TRACKING_URI'] = f'sqlite:///{OUT}/mlflow_colab.db'
os.environ['MLFLOW_ARTIFACT_ROOT'] = str(OUT / 'mlruns')
os.environ['OPTUNA_STORAGE'] = f'sqlite:///{OUT}/optuna_colab.db'
import pandas as pd
REG = 'trained_models/model_registry.csv'
rows_before = len(pd.read_csv(REG))
print('registry rows before:', rows_before)

## 7 · Sanity: the model tests pass on this machine

In [ ]:
!python -m pytest -q tests/test_models_torch.py tests/test_feature_pipeline.py

## 8 · TCN: default-config run, then 50-trial Optuna study + tuned model
The default run gives the `tcn` row (HANDOFF starting config). The study tunes channels, kernel size, dropout
and learning rate on **validation** RMSE only; the best settings then train `tcn_tuned`, scored once on test.

In [ ]:
!python -m src.train_tcn --device cuda --model-name tcn

In [ ]:
!python -m src.train_tcn_optuna --device cuda --n-trials 50

## 9 · Package the new outputs for the local repo
Creates `pm-outputs/colab_outputs_<time>.zip` on Drive. Download it and, in the local repo, run
`python scripts/import_colab_outputs.py <zip>` — it appends the new registry rows (never overwriting local ones).

In [ ]:
import zipfile, time, subprocess
reg = pd.read_csv(REG)
new_rows = reg.iloc[rows_before:]
print(new_rows[['model_name', 'version', 'test_rmse', 'nasa_score', 'git_commit']])
stamp = time.strftime('%Y%m%d_%H%M%S')
zpath = OUT / f'colab_outputs_{stamp}.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    z.writestr('registry_new_rows.csv', new_rows.to_csv(index=False))
    for p in new_rows['path']:
        for f in pathlib.Path(p).rglob('*'):
            if f.is_file(): z.write(f, f.as_posix())
    deep = ('tcn', 'lstm')                        # only deep-model files - never touch local xgboost/rocket configs
    for f in [f for d in ('configs', 'results') for f in pathlib.Path(d).glob('*') if f.name.startswith(deep)]:
        z.write(f, f.as_posix())
print('saved', zpath, f'({zpath.stat().st_size / 1e6:.1f} MB)')